# 03 · Inference-time scaling

Companion notebook for [Chapter 4](../chapters/ch04-inference-scaling.md).

**You will:** turn logits into samples, implement top-p, and run self-consistency —
then check whether voting actually helped.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, QWEN_CONFIG_06_B, Qwen3Tokenizer, download_qwen3_small,
)

download_qwen3_small(kind="base", out_dir="qwen3")     # 1.5 GB, cached after the first run

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=Path("qwen3") / "tokenizer-base.json"
)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(Path("qwen3") / "qwen3-0.6B-base.pth"))
model.to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 1 · Temperature, from the logits up

Softmax is $$p_i = e^{z_i}/\sum_j e^{z_j}$$. Dividing the logits by a temperature
$$T$$ rescales the gaps: $$T<1$$ sharpens, $$T>1$$ flattens.

In [ ]:
from reasoning_from_scratch.ch04 import scale_logits_by_temperature

logits = torch.tensor([[4.0, 2.0, 1.0, 0.5, 0.0]])

for T in (0.5, 1.0, 2.0, 5.0):
    scaled = scale_logits_by_temperature(logits, T)
    probas = torch.softmax(scaled, dim=-1)
    top = probas.max().item()
    print(f"T={T:<4} top-token prob={top:.3f}  dist={[round(p,3) for p in probas[0].tolist()]}")

**Read the numbers.** At T=0.5 the top token takes most of the mass. At T=5.0 the
distribution is nearly uniform. Greedy decoding is the limit T→0.

## 2 · Top-p (nucleus) filtering

Keep the smallest set of tokens whose cumulative probability reaches `p`. The
subtlety: the comparison uses the cumulative mass **before** each token, so the
token that crosses the threshold is kept.

In [ ]:
from reasoning_from_scratch.ch04 import top_p_filter

probas = torch.tensor([[0.50, 0.41, 0.09]])
for p in (0.9, 0.5, 0.95):
    out = top_p_filter(probas, p)
    kept = (out > 0).sum().item()
    print(f"top_p={p}: kept {kept} token(s) -> {[round(v,3) for v in out[0].tolist()]}")

With `top_p=0.9` and `[0.5, 0.41, 0.09]`, **two** tokens survive: prefix mass before
the second token is 0.5 < 0.9, so it is kept. A naive `cumsum < top_p` would drop
it.

## 3 · Sampling from the model

**Predict first:** with `temperature=0.9, top_p=0.9`, will two samples from the
same prompt differ? By how much?

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex, generate_text_top_p_stream_cache,
)

raw = "Half the value of $3x-9$ is $x+37$. What is the value of $x$?"
prompt = render_prompt(raw)

for seed in (0, 1, 2):
    torch.manual_seed(seed)
    answer = generate_text_stream_concat_flex(
        model, tokenizer, prompt, device, max_new_tokens=512,
        generate_func=generate_text_top_p_stream_cache,
        temperature=0.9, top_p=0.9,
    )
    print(f"--- seed {seed} ---")
    print(answer[-300:])

**Look at the tails.** Same question, same reasoning structure, often a different
final answer. That variance is what voting exploits — and also what limits it: if
the samples are correlated, voting over them adds less than you hope.

## 4 · Self-consistency

In [ ]:
from reasoning_from_scratch.ch03 import grade_answer, load_math500_test
from reasoning_from_scratch.ch04 import self_consistency_vote

math500 = load_math500_test()

for item in math500[:3]:                     # TRY: 5 items, then 20
    res = self_consistency_vote(
        model, tokenizer, render_prompt(item["problem"]), device,
        num_samples=5, temperature=0.9, top_p=0.9, max_new_tokens=2048, seed=0,
    )
    print(f"votes    : {res['counts']}")
    print(f"winner   : {res['final_answer']!r}")
    print(f"tie?     : {res['majority_winners'] if res['final_answer'] is None else 'no'}")
    print(f"correct  : {grade_answer(str(res['final_answer']), item['answer'])}")
    print(f"any right: {[grade_answer(s, item['answer']) for s in res['short_answers']]}")
    print("---")

## 5 · Did voting help?

The published ladder says sampling alone is worth very little (+2.2 points) and a
CoT prompt is worth a lot (+25 points). Check whether that holds for you.

In [ ]:
def eval_subset(n_questions=5, num_samples=1, cot=False):
    correct = 0
    for item in math500[:n_questions]:
        raw = item["problem"] + ("\n\nExplain step by step." if cot else "")
        if num_samples == 1:
            torch.manual_seed(0)
            answer = generate_text_stream_concat_flex(
                model, tokenizer, render_prompt(raw), device, max_new_tokens=1024,
                generate_func=generate_text_top_p_stream_cache,
                temperature=0.9, top_p=0.9,
            )
            from reasoning_from_scratch.ch03 import extract_final_candidate
            pred = extract_final_candidate(answer, fallback="number_then_full")
        else:
            res = self_consistency_vote(
                model, tokenizer, render_prompt(raw), device,
                num_samples=num_samples, temperature=0.9, top_p=0.9,
                max_new_tokens=1024, seed=0, show_progress=False,
            )
            pred = str(res["final_answer"])
        correct += bool(grade_answer(pred, item["answer"]))
    return correct / n_questions

# TRY: comment these in once the cells above run fast enough for you
# print("greedy-ish      :", eval_subset(5, 1, cot=False))
# print("with CoT        :", eval_subset(5, 1, cot=True))
# print("SC n=3 + CoT    :", eval_subset(5, 3, cot=True))

**Reference numbers (full 500, from the repo):** greedy 15.6% · CoT 40.6% ·
top-p only 17.8% · SC n=3 29.6% · SC n=10 31.6% · SC n=10 + CoT 52.0%.

Five questions cannot reproduce those; you are looking for the *ordering*, not the
values.

## 6 · Exercises

1. Why is SC n=5 (27.8%) worse than n=3 (29.6%) without CoT? Look at
   `short_answers` for a case where voting went wrong.
2. Write a tie-breaker for `final_answer is None` using average log-probability.
3. Sweep temperature over (0.5, 0.9, 1.2) on five questions. Where does it break?